In [9]:
from pycaret.datasets import get_data
from google.colab import drive
#drive.mount('/content/drive')
from datetime import *
import os
import shutil
import pandas as pd
import numpy as np

%pip install --upgrade gspread #IMPORTANT

import gspread
print(gspread.__version__) # make sure gspread is of the latest version
from google.colab import auth
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Constants
global WORKING_DIRECTORY,RAWDATAFILES,SUPPORTED_FILEFORMATS,RAW_Data_DIR,MERGE_FILES

SUPPORTED_FILEFORMATS={"CSV":pd.read_csv,"TXT":pd.read_fwf,"JSON":pd.read_json,"XML":pd.read_xml,"XLSX":pd.read_excel,"XLS":pd.read_excel}
COL_Prefixes ={"HeartRate":"HR_","Metadata":"MD_","WatchAccelerometer":"ACC_", "WatchGravity":"G_","WatchGyroscope":"GYRO_","WatchOrientation":"OR_","WatchTotalAcceleration":"TACC_"}

RAWDATAFILES={}

MERGE_FILES=1


# Get the current working directory
cwd = os.getcwd()
WORKING_DIRECTORY = os.path.dirname(os.path.dirname(cwd)) #Parent directory

print("The current working directory is:", WORKING_DIRECTORY)

#DATA_DIRECTORY = os.path.join(WORKING_DIRECTORY,"Project\\Data")
#print("The data directory is:", DATA_DIRECTORY)
DATA_DIRECTORY=  '/content/drive/MyDrive/MLTraining/'

RAW_Data_DIR=  '/content/drive/MyDrive/MLTraining/'
PREPROCESS_Data_DIR=  os.path.join(DATA_DIRECTORY,"02-preprocessed")
FEATURES_Data_DIR=  os.path.join(DATA_DIRECTORY,"03-features")
PREDICTIONS_Data_DIR=  os.path.join(DATA_DIRECTORY,"04-predictions")

#import previous modules output
# ts_df= pd.read_pickle(os.path.join(RAW_Data_DIR,"timeseries_data.pkl"))
# exercise_df = pd.read_pickle(os.path.join(RAW_Data_DIR,"exercise_data.pkl"))
# training_df= pd.read_pickle(os.path.join(RAW_Data_DIR,"training_data.pkl"))
# mw_detecxtions=pd.read_pickle(os.path.join(RAW_Data_DIR,"moving_window_detections.pkl"))
# peaks_df = pd.read_pickle(os.path.join(RAW_Data_DIR,"peaks.pkl"))
# valleys_df=pd.read_pickle(os.path.join(RAW_Data_DIR,"valleys.pkl"))


worksheet = gc.open("Copy of Training Notes").sheet1
# get_all_values gives a list of rows.
rows = worksheet.get_all_values()

train_df = pd.DataFrame.from_records(rows)
train_df


6.2.1
The current working directory is: /


,0,1,2,3,4
0,exercise,weight_kg,repetitions,Time,Date
1,warmup,,,,5.11.2025
2,abductor,32,"12,12,12",,5.11.2025
3,leg extension,"68,70,70",12;10;8,,5.11.2025
4,adductor,"30,32,32","12,11,8",,5.11.2025
...,...,...,...,...,...
181,reversed fly,52,"12,8, 6",14:34,19.12.2025
182,lat divergent,45,"12, 12,11",14:45,19.12.2025
183,bench pullover,20,"12, 12, 12",13:02,19.12.2025
184,decline abs crunch,,"30,30,40",13:12,19.12.2025


In [10]:
#filename = '/content/drive/MyDrive/MLTraining/2025-12-12_12-32-23/WatchGyroscope.csv'

#refurbished from 1. module
def loadfile(filename):
    """
    Arguments
    filename: must be full unc path
    """
    try:
        global RAWDATAFILES, MERGE_FILES
        parent_folder= os.path.abspath(os.path.join(filename, os.pardir)).split("\\")[-1]
        base_name = os.path.basename(filename).split(".")[0]
        print(base_name)
        #Choose respective reader according to filetype
        df=SUPPORTED_FILEFORMATS[filename.upper().split(".")[-1]](filename)
        #df.info()
        if MERGE_FILES==1:
            if base_name in RAWDATAFILES:
                RAWDATAFILES[base_name] = pd.concat([RAWDATAFILES[base_name], df.copy()], axis=0,join="outer")
            else:
                RAWDATAFILES[base_name] = df.copy()
        else:
            #Append the data frame to the global dictionary for further usage
            RAWDATAFILES[parent_folder + ":" + base_name] = df.copy()

        print(f"File {base_name} successfully loaded.")

    except Exception as e:
        print(f"Could not load file: {filename}. Error: {e}")

#Load all files
for folder in [f.path for f in os.scandir(RAW_Data_DIR) if f.is_dir()]:
    source = os.listdir(folder)
    print(source)
    filteredlist = [x for x in source if x.upper().split(".")[-1] in list(SUPPORTED_FILEFORMATS.keys()) ]
    for idx, file in enumerate(filteredlist):
        print(file)
        loadfile(os.path.join( folder,file))

#Rename columns
for k in RAWDATAFILES:
    RAWDATAFILES[k].columns = [COL_Prefixes[k] + a for a in RAWDATAFILES[k].columns]

['HeartRate.csv', 'WatchGyroscope.csv', 'Annotation.csv', 'WatchAccelerometer.csv', 'WatchOrientation.csv', 'WatchTotalAcceleration.csv', 'WatchGravity.csv', 'Metadata.csv']
HeartRate.csv
HeartRate
File HeartRate successfully loaded.
WatchGyroscope.csv
WatchGyroscope
File WatchGyroscope successfully loaded.
Annotation.csv
Annotation
Could not load file: /content/drive/MyDrive/MLTraining/2025-12-12_14-11-22/Annotation.csv. Error: No columns to parse from file
WatchAccelerometer.csv
WatchAccelerometer
File WatchAccelerometer successfully loaded.
WatchOrientation.csv
WatchOrientation
File WatchOrientation successfully loaded.
WatchTotalAcceleration.csv
WatchTotalAcceleration
File WatchTotalAcceleration successfully loaded.
WatchGravity.csv
WatchGravity
File WatchGravity successfully loaded.
Metadata.csv
Metadata
File Metadata successfully loaded.
['Metadata.csv', 'WatchGravity.csv', 'WatchAccelerometer.csv', 'WatchOrientation.csv', 'WatchTotalAcceleration.csv', 'HeartRate.csv', 'Annotatio

In [11]:

RAW_DF = pd.DataFrame([pd.to_datetime("2025-01-01 00:00:00")],columns=["time"])
RAW_DF.set_index(["time"])

MD_DF = pd.DataFrame()
#print(RAWDATAFILES)
for element in RAWDATAFILES:
    #RAWDATAFILES[element].info()
    if element!="Metadata":
        RAWDATAFILES[element]["time"] = pd.to_datetime(RAWDATAFILES[element][COL_Prefixes[element]+"time"])
        RAWDATAFILES[element].set_index("time", inplace=True)
        RAW_DF= RAW_DF.join(RAWDATAFILES[element], how='outer')
    else:
        RAWDATAFILES[element]["time"] = pd.to_datetime(RAWDATAFILES[element][COL_Prefixes[element]+"recording epoch time"])
        RAWDATAFILES[element].set_index("time", inplace=True)
        MD_DF= RAWDATAFILES[element].copy()

data = RAW_DF

In [12]:

from pycaret.classification import ClassificationExperiment
s = ClassificationExperiment()
s.setup(data, target = train_df[0], session_id = 123)

ValueError: X and y don't have the same indices!